In [114]:
from pathlib import Path
import numpy as np
import pandas as pd
from pathlib import Path

# Adding protein abundance data
This paper has absolute protein abundance data for different E. coli strains in different conditions. The goal of this notebook is to extract protein abundance data and Ribo-Seq data for E. coli grown in glucose M9 (or similar) to check that these transporters are usually expressed


In [115]:
# Data

data_folder = Path('../../../data')
folder = data_folder / 'mori_2021'
fn2 = folder / 'msb20209536-sup-0003-datasetev2.xlsx' # Sample info
fn3 = folder / 'msb20209536-sup-0004-datasetev3.xlsx' # Sample info
fn6 = folder / '44320_2024_62_moesm1_esm.xlsx' # Data for "calibration samples on E. coli in MOPS medium"
fn8 = folder / '44320_2024_62_moesm3_esm.xlsx' # Data for samples listed in fn2
fn9 = folder / '44320_2024_62_moesm4_esm.xlsx' # Data for samples listed in fn3




selected_transporters_fn = data_folder / 'this_project/6_transporterKO' / 'A_selected_transporters.csv'
df_sel = pd.read_csv(selected_transporters_fn, sep = '\t', encoding='latin1')
# df_sel = pd.read_excel(selected_transporters, sheet_name='Table EV1A', skiprows=3, usecols='A:H').iloc[1:]


In [116]:
df6 = pd.read_excel(fn6, sheet_name='EV6-CalibrationSamplesProteins', header = [0,1])
df8 = pd.read_excel(fn8, sheet_name='EV8-AbsoluteMassFractions-1')
df9 = pd.read_excel(fn9, sheet_name='EV9-AbsoluteMassFractions-2')


In [117]:
df6.rename(columns={"Unnamed: 0_level_1":'',	"Unnamed: 1_level_1":'',	"Unnamed: 2_level_1":'',	"Unnamed: 3_level_1":'', "Unnamed: 4_level_1":''}, level = 1, inplace=True)
    

In [118]:
df6.columns = ['_'.join(col) if len(col[1]) else col[0] for col in df6.columns]

In [119]:
xtop1_cols = [x for x in df6.columns if 'xTop' in x]

In [120]:
xtop1_cols

['xTop_A1-1',
 'xTop_A1-2',
 'xTop_A1-3',
 'xTop_C1',
 'xTop_F1-1',
 'xTop_F1-2',
 'xTop_F1-3']

In [121]:
df6_drop = ['TopPep1_A1-1', 'TopPep1_A1-2', 'TopPep1_A1-3', 'TopPep1_C1',
       'TopPep1_F1-1', 'TopPep1_F1-2', 'TopPep1_F1-3', 'TopPep3_A1-1',
       'TopPep3_A1-2', 'TopPep3_A1-3', 'TopPep3_C1', 'TopPep3_F1-1',
       'TopPep3_F1-2', 'TopPep3_F1-3', 'iBAQ_A1-1', 'iBAQ_A1-2', 'iBAQ_A1-3',
       'iBAQ_C1', 'iBAQ_F1-1', 'iBAQ_F1-2', 'iBAQ_F1-3']
df6.drop(columns=df6_drop, inplace=True)

In [122]:
df6

,Gene name,Gene locus,Protein ID,Molecular weight (kDa),"Ribosome profiling mass fractions (Li et al., 2014)",xTop_A1-1,xTop_A1-2,xTop_A1-3,xTop_C1,xTop_F1-1,xTop_F1-2,xTop_F1-3
0,aaeA,b3241,P46482,34.742084,1.213766e-06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,aaeB,b3240,P46481,73.590614,1.028399e-06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,aaeR,b3243,P67662,34.515616,1.905250e-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,aaeX,b3242,P46478,7.846496,2.741290e-07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,aas,b2836,P31119,80.699063,4.510946e-05,0.000153,0.000166,0.000159,0.000155,0.000163,0.000153,0.000158
...,...,...,...,...,...,...,...,...,...,...,...,...
4337,zraR,b4004,P14375,48.393976,1.014429e-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4338,zraS,b4003,P14377,51.031193,1.426280e-06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4339,zupT,b3040,P0A8H3,26.484489,4.015693e-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4340,zur,b4046,P0AC51,19.253992,2.246707e-05,0.000033,0.000000,0.000032,0.000032,0.000032,0.000037,0.000029


In [123]:
gene_locus_to_weight = df6.set_index('Gene locus')['Molecular weight (kDa)'].to_dict()

In [124]:
df6['xTop_A1-1'].sum()

np.float64(1.0000000000000018)

In [125]:
# Convert from mass fractions to number fractions
total_mass = {x: np.sum(df6[x]/df6['Molecular weight (kDa)']) for x in xtop1_cols + ['Ribosome profiling mass fractions (Li et al., 2014)']}
df6n = df6.iloc[:, :4].copy()
for key, value in total_mass.items():
    df6n[key] = (df6[key]/df6['Molecular weight (kDa)'])/value

In [126]:
df6n.rename(columns={"Ribosome profiling mass fractions (Li et al., 2014)": "Ribosome profiling number fraction (Li et al., 2014)"}, inplace=True)

In [127]:
selected_cols_df8 = ["Lib-24", "Lib-25", "Lib-26", "Lib-27", "Lib-28", "Lib-29", "Lib-30", "Lib-06"] # "Lib-00-A1", "Lib-00-A2", "Lib-00-A3", "Lib-00-B1", "Lib-00-B2", "Lib-00-B3"

selected_cols_df9 = ["A1-1", "A1-2", "A1-3", "C1", "F1-1", "F1-2", "F1-3", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "D6", "D7", "D8", "F4", "F5", "F6", "F7", "F8"]

In [128]:
df8n = df8.iloc[:, :3].copy()
df9n = df9.iloc[:, :3].copy()

In [129]:
#Convert df8 to number fractions
df8['Molecular weight (kDa)'] = df8['Gene locus'].map(gene_locus_to_weight)
df9['Molecular weight (kDa)'] = df9['Gene locus'].map(gene_locus_to_weight)

total_mass_8 = {x: np.sum(df8[x]/df8['Molecular weight (kDa)']) for x in selected_cols_df8}
total_mass_9 = {x: np.sum(df9[x]/df9['Molecular weight (kDa)']) for x in selected_cols_df9}
for key, value in total_mass_8.items():
    df8n[key] = (df8[key]/df8['Molecular weight (kDa)'])/value
    
for key, value in total_mass_9.items():
    df9n[key] = (df9[key]/df9['Molecular weight (kDa)'])/value



In [130]:
df8_all =  ['Lib-01', 'Lib-02', 'Lib-03','Lib-04', 'Lib-05', 'Lib-06', 'Lib-07', 'Lib-08', 'Lib-09', 'Lib-10',
       'Lib-11', 'Lib-12', 'Lib-13', 'Lib-14', 'Lib-15', 'Lib-16', 'Lib-17',
       'Lib-18', 'Lib-19', 'Lib-20', 'Lib-21', 'Lib-22', 'Lib-23', 'Lib-24',
       'Lib-25', 'Lib-26', 'Lib-27', 'Lib-28', 'Lib-29', 'Lib-30']

In [131]:
df9_all = ['A1-1', 'A1-2', 'A1-3', 'C1',
       'F1-1', 'F1-2', 'F1-3', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'D6',
       'D7', 'D8', 'F4', 'F5', 'F6', 'F7', 'F8', 'D1', 'D2', 'D3', 'D4', 'D5',
        'F2', 'F3', 'A2', 'E1', 'E2', 'E3', 'E4', 'H1', 'H5']

In [132]:
mean_protein_number_fraction = []
std_protein_number_fraction = []
detected_in_all_c_limit = []
detected_in_all = []
ribo_seq = []
for i, row in df_sel.iterrows():
    try:
        idx6 = np.where(df6n['Gene locus']==row['Blattner ID'])[0][0]
        idx8 = np.where(df8n['Gene locus']==row['Blattner ID'])[0][0]
        idx9 = np.where(df8n['Gene locus']==row['Blattner ID'])[0][0]
    except IndexError:
        mean_protein_number_fraction.append(None)
        std_protein_number_fraction.append(None)
        detected_in_all_c_limit.append(None)
        detected_in_all.append(None)
        ribo_seq.append(None)
    else:
        values6 = df6n.loc[idx6, xtop1_cols].values
        values8 = df8n.loc[idx8, selected_cols_df8].values
        values9 = df9n.loc[idx9, selected_cols_df9].values
        values = np.concatenate([values6, values8, values9])
        mean_protein_number_fraction.append(np.mean(values))
        std_protein_number_fraction.append(np.std(values))
        detected_in_all_c_limit.append(np.sum(values>0)/len(values))

        # Check how many datasets detect the protein when considering all datasets
        all_values = np.concatenate([df6.loc[idx6, xtop1_cols].values, 
                                     df8.loc[idx8, df8_all].values,
                                     df9.loc[idx9, df9_all].values])
        detected_in_all.append(np.sum(all_values>0)/len(all_values))
            
        ribo_seq.append(df6n.loc[idx6, 'Ribosome profiling number fraction (Li et al., 2014)'])

In [133]:
len(values)

37

In [134]:
df_sel['Mean protein number fraction (C-lim)'] = mean_protein_number_fraction
df_sel['Std protein number fraction (C-lim)'] = std_protein_number_fraction
df_sel['Detected in fraction of conditions (all)'] = detected_in_all
df_sel['Detected in fraction of conditions (C-lim)'] = detected_in_all_c_limit
df_sel['Ribosome profiling number fraction (Li et al., 2014)'] = ribo_seq

In [135]:
df_sel.tail()

,JW ID,Blattner ID,Gene Name,Annotation,Growth rate in glucose + AA medium,Location,Transported metabolites,Metabolite class,Class,Mechanism/specific type,...,Expected direction,TCID,biocyc link,Comment,Paper,Mean protein number fraction (C-lim),Std protein number fraction (C-lim),Detected in fraction of conditions (all),Detected in fraction of conditions (C-lim),"Ribosome profiling number fraction (Li et al., 2014)"
62,JW5735,b4138,dcuA,C4-dicarboxylate antiporter,0.78,Inner membrane,"L-aspartate, fumarate, succinate, malate",Organic acid,C4-dicarboxylate uptake (Dcu) family,Succinate/fumarate antiport,...,Both,NaN,https://biocyc.org/gene?orgid=ECOLI&id=EG11225,DcuA is a C4-dicarboxylate transporter which i...,NaN,0.000103,0.000042,1.0,1.0,0.000102
63,JW4038,b4077,gltP,glutamate/aspartate:proton symporter,0.86,Inner membrane,"L-aspartate, L-glutamate",Amino acid,dicarboxylate/amino acid:cation symporter (DAA...,Proton symport,...,Import,NaN,https://biocyc.org/gene?orgid=ECOLI&id=EG10405,GltP accounts for approximately 60% of the tot...,NaN,0.000000,0.000000,0.0,0.0,0.000022
64,JW0009,b0010,satP,acetate/succinate:H+ï¿½symporter,0.64,Inner membrane,"Acetate, succinate",Organic acid,Acetate Uptake Transporter (AceTr),Proton symport,...,Both,NaN,https://biocyc.org/gene?orgid=ECOLI&id=EG11512,NaN,NaN,0.000000,0.000000,0.0,0.0,0.000007
65,JW2910,b2943,galP,D-galactose transporter,0.84,Inner membrane,"Galactose, D-glucose",Sugar,MFS,Proton symport,...,Import,NaN,https://biocyc.org/gene?orgid=ECOLI&id=EG12148,NaN,NaN,0.000000,0.000000,0.0,0.0,0.000034
66,JW1652,b1660,punC,Purine transporter,0.76,Inner membrane,Purines,Nucleobase,MFS,Unknown,...,Export,NaN,https://ecocyc.org/gene?orgid=ECOLI&id=YDHC-MO...,While the punC deletion in general has a negat...,https://pubmed.ncbi.nlm.nih.gov/34413462/,0.000000,0.000000,0.0,0.0,0.000007


In [136]:
df_sel['log10(Mean protein number fraction)'] = np.log10(df_sel['Mean protein number fraction (C-lim)'])
df_sel['log10(Ribosome profiling)'] = np.log10(df_sel['Ribosome profiling number fraction (Li et al., 2014)'])

/Users/snorre/miniconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [137]:
# df_sel.drop(columns=['Location in operon'], inplace=True)
df_sel.replace(-np.inf, np.nan, inplace=True)

In [138]:
fn_new = data_folder / 'this_project' / '6_transporterKO' / 'B_selected_transporters_with_expression_levels.csv'
df_sel.to_csv(fn_new)

# Make transporter metadata table

# Shorter table with only key information

In [139]:
keep_cols = ['JW ID', 'Gene Name', 'Annotation', 'Location', 'Type', 'Expected direction','Metabolite class', 'Transported metabolites']
short_metadata_fn = data_folder / 'this_project' / '6_transporterKO' / 'C_transporters_short_metadata.csv'
df_sel[keep_cols].to_csv(short_metadata_fn, index=False)